In [1]:
import json
g_metadata = None
with open('genome_h.json', 'r') as fh:
    g_metadata = json.load(fh)
h_to_genome = {}
handle_to_genome = {}
for genome_id in g_metadata:
    h, handle_ref = g_metadata[genome_id]
    if h not in h_to_genome:
        h_to_genome[h] = genome_id
    else:
        print('!')
    if handle_ref not in handle_to_genome:
        handle_to_genome[handle_ref] = genome_id
    else:
        print('!')

In [57]:
def read_ani_out(f):
    ani_score = {}
    with open(f, 'r') as fh:
        l = fh.readline()
        while(l):
            _p = l.split('\t')
            g1, g2, score, k1, k2 = _p
            if g1 not in ani_score:
                ani_score[g1] = {}
            ani_score[g1][g2] = (score, k1, k2)
            l = fh.readline()
    return ani_score
ani_score_gca = read_ani_out('./ani_gtdb_gca2.txt')
ani_score_gcf = read_ani_out('./ani_gtdb_gcf2.txt')
ani_score_self = read_ani_out('./ani_self2.txt')

In [3]:
import networkx as nx
G = nx.Graph()

In [4]:
def get_gtdb_genome(s):
    return s.split('/')[-1][:-4]
def get_handle_name(handle_ref):
    return handle_to_genome[handle_ref.split('/')[-1]]

def load_data2(ani_scores, nodes, edges):
    for e1 in ani_scores:
        node1 = get_handle_name(e1)
        nodes.add(node1)
        for e2 in ani_scores[e1]:
            node2 = get_gtdb_genome(e2)
            nodes.add(node2)
            if e1 != e2:
                score, p1, p2 = ani_scores[e1][e2]
                p_1_2 = (node1, node2) 
                p_2_1 = (node2, node1) 
                if not p_1_2 in edges and not p_2_1 in edges:
                    edges[p_1_2] = [score]
                elif p_1_2 in edges:
                    edges[p_1_2].append(score)
                elif p_2_1 in edges:
                    edges[p_2_1].append(score)
                else:
                    print('this should never print')
nodes = set()
edges = {}
load_data2(ani_score_gca, nodes, edges)
print(len(nodes), len(edges))
load_data2(ani_score_gcf, nodes, edges)
print(len(nodes), len(edges))
# 33108 572044
# 124868 2571041

6006 30036
22090 105491


In [5]:
nodes_self = set()
for e1 in ani_score_self:
    node1 = get_handle_name(e1)
    nodes_self.add(node1)
    nodes.add(node1)
    for e2 in ani_score_self[e1]:
        node2 = get_handle_name(e2)
        nodes_self.add(node2)
        nodes.add(node2)
        if e1 != e2:
            score, p1, p2 = ani_score_self[e1][e2]
            p_1_2 = (node1, node2) 
            p_2_1 = (node2, node1) 
            if not p_1_2 in edges and not p_2_1 in edges:
                edges[p_1_2] = [score]
            elif p_1_2 in edges:
                edges[p_1_2].append(score)
            elif p_2_1 in edges:
                edges[p_2_1].append(score)
            else:
                print('this should never print')
            
            
print(len(nodes), len(edges)) #22177 107067

22177 107067


In [8]:
_edges = []
for p in edges:
    e1, e2 = p
    scores = edges[p]
    total = 0
    for s in scores:
        total += float(s)
    score = total / len(scores)
    _edges.append([e1, e2, score])

In [9]:
G = nx.Graph()
G.add_nodes_from(nodes)
G.add_weighted_edges_from(_edges)
print(G) #Graph with 22177 nodes and 107067 edges

In [18]:
centroids_nodes = set()
centroids_nodes.add('Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.8.contigs__.RAST')

In [19]:
from cobrakbase.kbaseapi_cache import KBaseCache
kbase = KBaseCache(path='/scratch/fliu/data/kbase/cache')
ws_id = 155805
ws_os = kbase.list_objects(ws_id)
for o in ws_os:
    if o[2].startswith('KBaseGenomes.Genome'):
        centroids_nodes.add(o[1])

In [14]:
def walk(graph, start, distance):
    nodes = set()
    _walk_collect(graph, start, distance, nodes)
    return nodes

def _walk_collect(graph, pos, distance, collect):
    collect.add(pos)
    for dst, attr in G[pos].items():
        v = attr['weight']
        #print(pos, dst, v, v >= distance)
        if dst not in collect:
            if v >= distance:
                _walk_collect(graph, dst, distance, collect)
walk(G, node_c, walk_distance)

{'0ac7d16ab31c8eba120d2a41d43560d19a771057d3a3f52195e0500deb82716c',
 'Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.metabat.51.contigs__.RAST',
 'Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_metabat.48.contigs__.RAST',
 'Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_metabat.50.contigs__.RAST',
 'Salt_Pond_MetaG_R1_C_D1_MG_DASTool_bins_maxbin.047.contigs__.RAST',
 'Salt_Pond_MetaG_R1_C_D2_MG_DASTool_bins_concoct_out.9.contigs.fa_assembly.RAST',
 'd25798567ea536d7fd8d0fb4d7e0f59a3208045508719ad634b223b866abaaf9'}

In [15]:
data = {}
for walk_distance in range(70,100):
    centroids = set(centroids_nodes)
    centroid_nodes = {}
    for n in G.nodes:
        if n in centroids:
            centroid_nodes[n] = {}
    print(len(centroid_nodes))
    visited = set()
    clusters = []
    for node_c in centroid_nodes:
        if node_c not in visited:
            visits = walk(G, node_c, walk_distance)
            visited |= visits
            clusters.append(visits)
    data[walk_distance] = {
        'clusters': clusters,
        'visited': visited
    }

310
310
310
310
310
310
310
310
310
310
310
310
310
310
310
310
310
310
310
310
310
310
310
310
310
310
310
310
310
310


In [16]:
distance_singletons = {}
for walk_distance in range(70,100):
    d = data[walk_distance]
    singletons = set()
    for c in d['clusters']:
        sz = len(c)
        centroids_in_c = centroids | c
        if sz == 1:
            singletons |= c
        elif centroids_in_c == c:
            #print(c)
            pass
    distance_singletons[walk_distance] = singletons
        #if c in centroids
    print(walk_distance, len(d['clusters']), len(d['visited']), len(singletons))

70 137 22177 39
71 137 22177 39
72 137 22177 39
73 137 22177 39
74 137 22177 39
75 150 21566 44
76 167 19546 48
77 175 12809 53
78 199 3720 71
79 219 2211 97
80 240 1630 121
81 249 1360 137
82 260 1157 144
83 263 1055 148
84 266 999 151
85 268 944 159
86 278 895 170
87 282 815 175
88 284 747 179
89 286 714 182
90 287 672 187
91 288 639 192
92 290 605 197
93 292 591 200
94 293 585 203
95 298 582 210
96 301 575 215
97 309 539 231
98 310 486 250
99 310 367 283


In [6]:
all_genomes = set()
for i in data:
    for c in data[i]['clusters']:
        all_genomes |= c

NameError: name 'data' is not defined

In [18]:
import json

In [27]:
with open('/scratch/fliu/data/gtdb/metadata_gca.json', 'r') as fh:
    metadata_gtdb_gca = json.load(fh)
with open('/scratch/fliu/data/gtdb/metadata_gcf.json', 'r') as fh:
    metadata_gtdb_gcf = json.load(fh)

In [28]:
with open('/scratch/fliu/data/gtdb/metadata_genome_gene_calls.json', 'r') as fh:
    metadata_gene_cals_path = json.load(fh)

In [23]:

import os
genomes = {}
assemblies = {}
genomes_contigs = {}


In [41]:
ws_ids = [155805, 163264]
for ws_id in ws_ids:
    ws_os = kbase.list_objects(ws_id)
    print(len(ws_os))
    for o in ws_os:
        if o[2].startswith('KBaseGenomes.Genome'):
            genome = kbase.get_from_ws(o[1], ws_id)
            if genome.info.id not in genomes:
                genomes[genome.info.id] = genome
            else:
                print('!!', o)

679
!! [3, 'Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.8.contigs__.RAST', 'KBaseGenomes.Genome-11.1', '2023-09-02T02:08:15+0000', 1, 'cliff_bueno', 155805, 'cliff_bueno:narrative_1693612235241', 'db630a36d897caba142685e9e010ae30', 9852570, None]
!! [4, 'Salt_Pond_MetaG_R1_B_D1_MG_DASTool_bins_metabat.20.contigs__.RAST', 'KBaseGenomes.Genome-11.1', '2023-09-02T02:08:27+0000', 1, 'cliff_bueno', 155805, 'cliff_bueno:narrative_1693612235241', '18975ce8d06d8eeaa4f262ec7f642498', 12725637, None]
!! [5, 'Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_concoct_out.4.contigs__.RAST', 'KBaseGenomes.Genome-11.1', '2023-09-02T02:08:44+0000', 1, 'cliff_bueno', 155805, 'cliff_bueno:narrative_1693612235241', 'd0d1dbef3a80097cc8c21b4ccf56b85f', 19689749, None]
!! [6, 'Salt_Pond_MetaG_R2_restored_DShore_MG_DASTool_bins_concoct_out.46.contigs__.RAST', 'KBaseGenomes.Genome-11.1', '2023-09-02T02:08:56+0000', 1, 'cliff_bueno', 155805, 'cliff_bueno:narrative_1693612235241', '9a3594112c7f08ed5237d985e9cb3220',

KeyboardInterrupt: 

In [49]:
ws_ids = [155805, 163264]
for ws_id in ws_ids:
    ws_os = kbase.list_objects(ws_id)
    print(len(ws_os))
    for o in ws_os:
        if o[2].startswith('KBaseGenomes.Genome'):
            if o[1] not in genomes:
                genome = kbase.get_from_ws(o[1], ws_id)
                genomes[o[1]] = genome

679
283


In [47]:
len(genomes)

310

In [13]:
all_genomes = set(G.nodes)

In [22]:
genome_to_file = {}
for k in all_genomes:
    gca = None
    gcf = None
    if k in metadata_gtdb_gca['h_to_genome']:
        gca = metadata_gtdb_gca['h_to_genome'][k]
    if k in metadata_gtdb_gcf['h_to_genome']:
        gcf = metadata_gtdb_gcf['h_to_genome'][k]
    if gca is None and gcf is None:
        #print(k, gca, gcf)
        pass
    elif (not gca is None and gcf is None) or (gca is None and not gcf is None):
        d = None
        if gca is None:
            d = set(gcf)
        else:
            d = set(gca)
        if len(d) == 1:
            pass
        else:
            print(k, gca, gcf)
    else:
        print(k, gca, gcf)

be0910c9b4c627d961c67e554a99cbca28d1883aba6879be2bb1847d93b59c5c None ['GCF_000484545.2_ASM48454v1', 'GCF_000484545.1_ASM48454v1']
8abdc76e82ffe147c478a1197ed8bb0308d43485324447348bf4ff8577077a46 None ['GCF_000373665.1_ASM37366v1', 'GCF_009864915.1_ASM986491v1']
2b99071d1d10a977c75bad558955d2b692b69efd93822a300546756cd4bb7d21 ['GCA_004329695.1_ASM432969v1'] ['GCF_001879695.1_ASM187969v1']
350ab7cb6f52a8384a55cb2ec00a47ef5aaa840c18bd5825a8cb693fee8c3998 None ['GCF_003412125.1_ASM341212v1', 'GCF_003412125.2_ASM341212v2']
760ed8f47bfa80ed9645848216c098a1d7c82efb7efca1236bb0a4ae7346c29a None ['GCF_000484495.1_ASM48449v1', 'GCF_000484495.2_ASM48449v1']
f8a487845c17acba04be0bc844e88f8cdd7192a1ce3206947d7eff9576bf786d None ['GCF_003411885.1_ASM341188v1', 'GCF_003411885.2_ASM341188v2']
e22eda489859c18a022f81554cfaef58c73a8f62deea6cf57b23746b9d1fac3f None ['GCF_001853325.1_ASM185332v1', 'GCF_001481455.1_ASM148145v1']
cef53a8758a384fe24d1a2530948f14d88f2217b38c93a0acb3e9e906c2bb39f None ['GCF_00

In [48]:
import gzip
base_folder = '/scratch/fliu/data/gtdb/genomes/global/cfs/cdirs/kbase/jungbluth/Projects/Project_Pangenome_GTDB/GTDB_r214_by_spcluster/'
data = None
with gzip.open(base_folder + '/' + 's__Guyparkeria_sp001641825--RS_GCF_001641825.1/GCF_001641825.1_protein.faa.gz', 'rb') as fh:
    data = fh.read()

In [50]:
faa_data = data.decode('utf-8')

In [53]:
print(faa_data[:1000])

>NZ_LRFH01000001.1_1 # 364 # 1041 # -1 # ID=1_1;partial=00;start_type=ATG;rbs_motif=GGA/GAG/AGG;rbs_spacer=5-10bp;gc_cont=0.659
MSATREPAIVLLSGGLDSATLAWMARADGYAVHALSFDYGQRHRAELAAAGRVAQAAGAV
EHRTIRIDMDGITGSSLTDVERSVPDYDPAQSEIPSTYVPARNLTFLSFALGWAEVLSAR
RIYIGVNAVDYSGYPDCRPAFIEKFQELANVATAAAHPFEIHAPLLHMTKAEIIRTGQAL
GVDYGLTVSCYRADEAGRACGRCDSCHYRQQGFRDAGVPDPTRYA*
>NZ_LRFH01000001.1_2 # 1044 # 1742 # -1 # ID=1_2;partial=00;start_type=ATG;rbs_motif=None;rbs_spacer=None;gc_cont=0.655
MSEAIAAGTPSLDQMGDRSTRSLRITEIFRSLQGESDHIGWPTVFVRLTGCPLRCVYCDT
AYAFTGGERMSPESILDRVAELDTRHVCVTGGEPLAQPGCIDLLRRLADADHEVSLETSG
AMDVAEVDPRVHVVMDLKTPSSGEETRNRWENLAHLKSSDEIKVVIGSREDFDWAVARVR
EHDLTRRFAVLFSPVFEGVSPRTLAEWILDSGLDIRFQMQLHKLIWGDEPGR*
>NZ_LRFH01000001.1_3 # 1739 # 2689 # -1 # ID=1_3;partial=00;start_type=ATG;rbs_motif=AGGA;rbs_spacer=5-10bp;gc_cont=0.682
MSASVSLCRRGLPLAVGVFACSLSGAAMAQDAPGWLNWGGSSSEETAKTADERPAVRTMP
ADTTSASTAPVANAGEANQSGQAVLLQRLMQRVDSLERQMQSVRGQLSEQERRLQRQSED
MQEALEAARSQQSAAAGSSMSSAGAQPSTTAPSAPEQPSAP

In [57]:
last = 70
res = {}
res[last] = distance_singletons[last]
carry = set(res[last])
for walk_distance in range(last + 1, 100):
    res[walk_distance] = distance_singletons[walk_distance] - carry
    carry |= res[walk_distance]
ani_left_out = {}
for walk_distance in range(last, 100):
    for g in res[walk_distance]:
        ani_left_out[g] = walk_distance
for g in centroid_nodes:
    if g not in ani_left_out:
        ani_left_out[g] = 99
with open('ani_left_out.json', 'w') as fh:
    fh.write(json.dumps(ani_left_out))

In [20]:
genomes = {}
assembly_to_genome = {}
ws_id = 155805
ws_os = kbase.list_objects(ws_id)
for o in ws_os:
    if o[2].startswith('KBaseGenomes.Genome'):
        genome = kbase.get_from_ws(o[1], ws_id)
        genomes[genome.info.id] = genome
        assembly_to_genome[kbase.get_object_info(genome.assembly_ref).id] = genome.info.id

In [80]:
import pandas as pd

taxa = {}
df = pd.read_csv('./gtdb_table_arc.csv', index_col=0)
for as_id, d in df.iterrows():
    g_id = assembly_to_genome[as_id]
    if g_id not in taxa:
        taxa[g_id] = d['Classification']
df = pd.read_csv('./gtdb_table_bac.csv', index_col=0)
for as_id, d in df.iterrows():
    g_id = assembly_to_genome[as_id]
    if g_id not in taxa:
        taxa[g_id] = d['Classification']
len(taxa)

310

In [81]:
def _insert_tree_taxa(t, array):
    if len(array) > 0:
        if array[0] not in t:
            t[array[0]] = {}
        _insert_tree_taxa(t[array[0]], array[1:])
tree = {}
for c in centroid_nodes:
    if c in taxa:
        _p = taxa[c].split(';')
        last = _p[-1]
        if last.endswith('__') and len(last) == 3:
            _p = _p[:-1]
        _p.append(c)
        _insert_tree_taxa(tree, _p)
    else:
        #print(c)
        pass

In [82]:
len(tree)

2

In [83]:
with open('taxa_tree.json', 'w') as fh:
    fh.write(json.dumps(tree))

In [70]:
everything = {}
for k in data:
    _data = data[k]
    for v in _data['visited']:
        if v not in everything:
            everything[v] = None

In [31]:
base_dir = '/scratch/fliu/data/gtdb/genomes/global/cfs/cdirs/kbase/jungbluth/Projects/Project_Pangenome_GTDB/GTDB_r214_by_spcluster/'
base_cache_dir = '/scratch/fliu/data/kbase/cache/handle/'

In [72]:
base_dir = '/scratch/fliu/data/gtdb/genomes/global/cfs/cdirs/kbase/jungbluth/Projects/Project_Pangenome_GTDB/GTDB_r214_by_spcluster/'
base_cache_dir = '/scratch/fliu/data/kbase/cache/handle/'
for k in everything:
    if k in g_metadata:
        everything[k] = base_cache_dir + '/' +  g_metadata[k][1]
    else:
        p = get_path(k)
        if p:
            everything[k] = base_dir + '/' + p

set()
set()
set()
set()
set()
set()
set()
set()
{'s__Streptomyces_glauciniger--RS_GCF_900188405.1/GCF_009864915.1_protein.faa.gz', 's__Streptomyces_glauciniger--RS_GCF_900188405.1/GCF_000373665.1_protein.faa.gz'}
set()
{'s__CAIOYC01_sp903837135--GB_GCA_903837135.1/GCA_903880465.1_protein.faa.gz', 's__CAIOYC01_sp903837135--GB_GCA_903837135.1/GCA_903882015.1_protein.faa.gz'}
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
{'s__Halorubrum_sp001564205--GB_GCA_001564205.1/GCA_001564005.1_protein.faa.gz', 's__Halorubrum_sp001564205--GB_GCA_001564205.1/GCA_001564205.1_protein.faa.gz'}
set()
{'s__Burkholderia_mallei--RS_GCF_000011705.1/GCF_002900675.1_protein.faa.gz', 's__Burkholderia_mallei--RS_GCF_000011705.1/GCF_002921055.1_protein.faa.gz'}
set()
set()
set()
set()
set()
set()
set()
set()
{'s__D2472_sp003283765--GB_GCA_003283765.1/GCA_902511355.1_protein.faa.gz', 's__D2472_sp003283765--GB_GCA_003283765.1/GCA_003283765.1_protein.faa.gz'}
set()
set()
s

In [108]:
len(metadata_gene_cals_path)

402709

In [55]:
def get_path(h):
    fetch = set()
    if h in metadata_gtdb_gca['h_to_genome']:
        for s in metadata_gtdb_gca['h_to_genome'][h]:
            p = s.split('_')
            ncbi_id = p[0] + '_' +  p[1]
            if ncbi_id in metadata_gene_cals_path:
                fetch.add(metadata_gene_cals_path[ncbi_id])
    elif h in metadata_gtdb_gcf['h_to_genome']:
        for s in metadata_gtdb_gcf['h_to_genome'][h]:
            p = s.split('_')
            ncbi_id = p[0] + '_' +  p[1]
            if ncbi_id in metadata_gene_cals_path:
                fetch.add(metadata_gene_cals_path[ncbi_id])
    else:
        return None
    
    if len(fetch) == 1:
        return fetch.pop()
    else:
        #print(fetch)
        return None


In [59]:
missing = set()
capture = set()
h_to_cap = {}
for k in all_genomes:
    if k in genomes:
        capture.add(k)
        #print(k)
        pass
    else:
        filename = get_path(k)
        if filename:
            h_to_cap[k] = filename
            capture.add(k)
            #print(k)
            pass
        else:
            missing.add(k)
            pass
len(missing), len(capture)

(493, 21684)

In [66]:
from tqdm import tqdm
h_to_genome_o = {}
for h, f in tqdm(h_to_cap.items()):
    genome = MSGenome.from_fasta(base_dir + '/' + f)
    h_to_genome_o[h] = genome

100%|██████████| 21234/21234 [1:04:19<00:00,  5.50it/s] 


In [68]:
from modelseedpy.core import MSGenome
from modelseedpy_ext.super_faa import SuperFaa

In [69]:
sfaa = SuperFaa()
sfaa.genomes.update(genomes)
sfaa.genomes

{}

In [75]:
len(sfaa.annotated_genes)

TypeError: object of type 'NoneType' has no len()

In [76]:
import json
with open('./data/sfaa.genome_feature_hash.json', 'w') as fh:
    fh.write(json.dumps(sfaa.genome_feature_hash))

In [79]:
from modelseedpy_ext.mmseqs.mmseqs import MMseqs2
mmseqs = MMseqs2(set())
mmseqs.mmseqs_bin = '/opt/mmseqs2/15-6f452/bin/mmseqs'
mmseqs.path = '/home/fliu/cliff_mags/data/mmseqs/'
mmseqs.cluster(master_faa_path='/home/fliu/cliff_mags/data/super_faa.faa', threads=120)

['/opt/mmseqs2/15-6f452/bin/mmseqs', 'easy-cluster', '--threads', '120', '--min-seq-id', '0.0', '--cov-mode', '0', '-c', '0.8', '/home/fliu/cliff_mags/data/super_faa.faa', '/home/fliu/cliff_mags/data/mmseqs//output/mmseqs', '/home/fliu/cliff_mags/data/mmseqs//output']


CompletedProcess(args=['/opt/mmseqs2/15-6f452/bin/mmseqs', 'easy-cluster', '--threads', '120', '--min-seq-id', '0.0', '--cov-mode', '0', '-c', '0.8', '/home/fliu/cliff_mags/data/super_faa.faa', '/home/fliu/cliff_mags/data/mmseqs//output/mmseqs', '/home/fliu/cliff_mags/data/mmseqs//output'], returncode=0, stdout="Create directory /home/fliu/cliff_mags/data/mmseqs//output\neasy-cluster --threads 120 --min-seq-id 0.0 --cov-mode 0 -c 0.8 /home/fliu/cliff_mags/data/super_faa.faa /home/fliu/cliff_mags/data/mmseqs//output/mmseqs /home/fliu/cliff_mags/data/mmseqs//output \n\nMMseqs Version:                     \t6f45232ac8daca14e354ae320a4359056ec524c2\nSubstitution matrix                 \taa:blosum62.out,nucl:nucleotide.out\nSeed substitution matrix            \taa:VTML80.out,nucl:nucleotide.out\nSensitivity                         \t4\nk-mer length                        \t0\nTarget search mode                  \t0\nk-score                             \tseq:2147483647,prof:2147483647\nAlpha

In [137]:
feature_to_genome = {}
with open('/scratch/fliu/data/cliff/ani_mmseqs/master.faa', 'w') as fh:
    for h in everything:
        filename = everything[h]
        if filename:
            if 'handle' in filename:
                genome = MSGenome.from_fasta(filename)
                for f in genome.features:
                    f_id = f.id
                    if f.seq and len(f.seq) > 0:
                        if f.id not in feature_to_genome:
                            fh.write(f'>{f_id}\n')
                            fh.write(f'{f.seq}\n')
                            feature_to_genome[f.id] = h
                        else:
                            print('conflict', h, feature_to_genome[f.id])
            else:
                genome = MSGenome.from_fasta(filename)
                for f in genome.features:
                    s = f.id.split()
                    f_id = s[0]
                    if f.seq and len(f.seq) > 0:
                        if f_id not in feature_to_genome:
                            fh.write(f'>{f_id}\n')
                            fh.write(f'{f.seq}\n')
                            feature_to_genome[f_id] = h
                        else:
                            print('conflict', h, feature_to_genome[f_id])

In [ ]:
%ls 

In [138]:
with open('/scratch/fliu/data/cliff/metadata_ani_g.json', 'w') as fh:
    fh.write(json.dumps(feature_to_genome))

In [141]:
valid = set(feature_to_genome.values())
len(valid)

21684

In [217]:
member_limit = 50000
bad = set()
done = set()
similar_genomes = {k:set() for k in centroids}
for i in range(99, 70, -1):
    for cc in data[i]['clusters']:
        tt = cc & centroids
        others = cc - tt
        for _t in tt:
            if _t not in done:
                members = similar_genomes[_t]
                members_count = len(members)
                for _o in others:
                    if len(members) < member_limit and _o not in bad:
                        members.add(_o)
                    else:
                        done.add(_t)
                        break
                        
for k in similar_genomes:
    if len(similar_genomes[k]) == member_limit:
        print(k)

In [194]:
similar_genomes['Salt_Pond_MetaG_R2_C_H2O_MG_DASTool_bins_concoct_out.60.contigs__.RAST']

{'2a00b09cf8fa0a52c434fbde82e8a9a1d4e967aa340f8d7b93a5f90bc4b5cc81',
 '9f883fcf64824b3128f2b1cf1b982d29681693cb59e9b0c4ac116efebd1e771c',
 'Salt_Pond_MetaG_R2_A_H2O_MG_DASTool_bins_concoct_out.51.contigs.fa_assembly.RAST',
 'Salt_Pond_MetaG_R2_B_H2O_MG_DASTool_bins_concoct_out.4.contigs.fa_assembly.RAST',
 'be5cd4f063004e9b2b4cb0a2168fca6a6585ef25184a6b51f016f7ca46847d12',
 'cd3e1d1fcbae456da76bbfc6f3f15e538adcdbdd11c0bcc013bf8667ca9fe042',
 'd599b8ca462f0a4113b63ca5e3e792c2b6a398793d15cc972675a146ce01f08c',
 'dc8d423291da814508d79bf5e25e00b5a0c861579c7106f808a54fa2d52ff5c0',
 'e393c653dde4c5f7feb8d4465dbb2796796fbad404f523a279490d0d8a1e1fc8',
 'ede5a1e394f39e509742d01574f8121b1d2c2470ec68f97d86877bb503cc823f'}

In [218]:
import json
with open('/scratch/fliu/data/cliff/genome_clusters_70_no_limit.json', 'w') as fh:
    fh.write(json.dumps({x:list(y) for x, y in similar_genomes.items()}))


In [30]:
for a, b in similar_genomes.items():
    for i in b:
        print(i)

Salt_Pond_MetaG_R2_B_H2O_MG_DASTool_bins_concoct_out.4.contigs.fa_assembly.RAST
Salt_Pond_MetaG_R2_A_H2O_MG_DASTool_bins_concoct_out.51.contigs.fa_assembly.RAST
Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_metabat.17.contigs.fa_assembly.RAST
Salt_Pond_MetaGSF2_C_H2O_MG_DASTool_bins_metabat.17.contigs.fa_assembly.RAST
0cdf9c2a300fbc7087b2a1c8038ece05b179c1ba59200c5bdf0bcaf99396cc16
Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.15.contigs.fa_assembly.RAST
8f852dce27c8359e4370a08b4b8cda00bc74e4f35ddcdadcee372341b48f971b
Salt_Pond_MetaGSF2_B_D2_MG_DASTool_bins_concoct_out.21.contigs.fa_assembly.RAST
Salt_Pond_MetaGSF2_C_D1_MG_DASTool_bins_concoct_out.23.contigs.fa_assembly.RAST
b28ceb70256f7f336dadf040befffc0343d5539fadac26b2fb47726367339f2a
Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_metabat.13.contigs.fa_assembly.RAST
17ec6f93de2047f0780fa84bf51e78b87f202f244928362a75f0ef913f364c7a
4804ae89844ac6ac6672bd84dd9d09bb86df8258ace45b818660142b7b317a3c
c98dc8a15c60de0fed7fc7bb09d8e3af3b4ac2c0687d2374

In [32]:
data.keys()

dict_keys([70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99])

In [35]:
len(centroids)

310

In [ ]:
# load centroid annotation
# 

In [33]:
data[95]

{'clusters': [{'Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_metabat.50.contigs__.RAST'},
  {'Salt_Pond_MetaG_R2A_C_D2_MG_DASTool_bins_concoct_out.4.contigs__.RAST'},
  {'Salt_Pond_MetaG_R2A_C_D2_MG_DASTool_bins_metabat.38.contigs__.RAST'},
  {'Salt_Pond_MetaG_R2_restored_DShore_MG_DASTool_bins_concoct_out.12.contigs__.RAST'},
  {'711d29c2cffc4660070a645e7df7ed03f482d3990f2b1315542a93deaf26489c',
   'Salt_Pond_MetaGSF2_A_H2O_MG_DASTool_bins_metabat.28.contigs__.RAST',
   'Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_concoct_out.76.contigs.fa_assembly.RAST',
   'Salt_Pond_MetaGSF2_C_H2O_MG_DASTool_bins_concoct_out.34.contigs.fa_assembly.RAST',
   'Salt_Pond_MetaG_R2A_A_H2O_MG_DASTool_bins_concoct_out.78.contigs__.RAST',
   'Salt_Pond_MetaG_R2A_B_H2O_MG_DASTool_bins_concoct_out.59.contigs__.RAST'},
  {'Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.metabat.18.contigs__.RAST',
   'Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_concoct_out.44.contigs.fa_assembly.RAST',
   'Salt_Pond_MetaG_R1_B_D1_MG_DASTool_bins

In [36]:
import cobrakbase
from cobrakbase.kbaseapi_cache import KBaseCache
kbase = KBaseCache(path='/scratch/fliu/data/kbase/cache')

In [160]:
from tqdm import tqdm

In [161]:
for c in tqdm(centroids):
    a = catalog_genome(c, annotation_database)
    #c = 'Salt_Pond_MetaGSF2_A_D1_MG_DASTool_bins_concoct_out.23.contigs__.RAST'

100%|██████████| 310/310 [1:59:16<00:00, 23.09s/it]   


In [163]:
annotation_database['Valyl-tRNA synthetase (EC 6.1.1.9)']

{'Salt_Pond_MetaG_R2_C_H2O_MG_DASTool_bins_metabat.29.contigs__.RAST': {'self:Salt_Pond_MetaG_R2_C_H2O_MG_DASTool_bins_metabat.29.contigs__.RAST.CDS.9': 100.0,
  '67ffd794e4965354d34a33c725edd559d9e53fbcec0d51d9b93be7a00606e630:JALDAO010000030.1_3': 78.3769,
  '3feaa740925f54ad747eb523b39e53fd5bf7e68ed5f07a21534173394b63f30f:JALDAN010000028.1_3': 78.3115},
 'Salt_Pond_MetaG_R2_C_D1_MG_DASTool_bins_concoct_out.7.contigs__.RAST': {'self:Salt_Pond_MetaG_R2_C_D1_MG_DASTool_bins_concoct_out.7.contigs__.RAST.CDS.2070': 100.0},
 'Salt_Pond_MetaG_R2_C_H2O_MG_DASTool_bins_concoct_out.60.contigs__.RAST': {'self:Salt_Pond_MetaG_R2_C_H2O_MG_DASTool_bins_concoct_out.60.contigs__.RAST.CDS.1845': 100.0,
  'e393c653dde4c5f7feb8d4465dbb2796796fbad404f523a279490d0d8a1e1fc8:PXQZ01000017.1_5': 85.5502,
  'ede5a1e394f39e509742d01574f8121b1d2c2470ec68f97d86877bb503cc823f:PXRT01000009.1_16': 85.0809,
  '10777e2869c2f7afd78d43852408a4a5aa077a16a9cb3e471afdd9e8d12be259:JXAO01000003.1_218': 80.1492,
  '10777e28

In [164]:
with open('/scratch/fliu/data/cliff/annotation_ani_prob.json', 'w') as fh:
    fh.write(json.dumps(annotation_database))

In [181]:
with open('/scratch/fliu/data/cliff/annotation_ani_prob_gep_85.json', 'w') as fh:
    fh.write(json.dumps(annotation_database_geq_85))
with open('/scratch/fliu/data/cliff/annotation_ani_prob_lo_85.json', 'w') as fh:
    fh.write(json.dumps(annotation_database_lo_85))

In [126]:
from modelseedpy import RastClient

In [131]:
rast = RastClient()

In [132]:
rast.annotate_genome(genome)

[{'tool_name': 'kmer_search',
  'id': 'ADBF794E-BF75-11EE-8FEC-814819DF3448',
  'hostname': 'pecan',
  'execution_time': 1706622264.21463,
  'parameters': ['-a',
   '-g',
   200,
   '-m',
   5,
   '-d',
   '/opt/patric-common/data/kmer_metadata_v2',
   '-u',
   'http://redwood.cels.anl.gov:6100/query']},
 {'execute_time': 1706622264.4986,
  'hostname': 'pecan',
  'tool_name': 'annotate_proteins_similarity',
  'id': 'ADEAC4FA-BF75-11EE-B686-C8401CDF3448',
  'parameters': []}]

In [122]:
genome_path

'/scratch/fliu/data/kbase/cache/handle//KBH_7675027'

In [137]:
genome_path = a[1]
genome = a[0]


NODE_1102_length_16627_cov_2.45012 {'hypothetical protein'}
NODE_1135_length_16280_cov_1.93989 {'hypothetical protein'}
NODE_1146_length_16155_cov_1.78369 {'hypothetical protein'}
NODE_1311_length_14877_cov_1.96875 {'hypothetical protein'}
NODE_1325_length_14764_cov_1.74974 {'hypothetical protein'}
NODE_1397_length_14374_cov_1.8719 {'hypothetical protein'}
NODE_1416_length_14284_cov_1.87667 {'hypothetical protein'}
NODE_1443_length_14108_cov_2.58615 {'hypothetical protein'}
NODE_1504_length_13731_cov_1.62577 {'hypothetical protein'}
NODE_1535_length_13512_cov_2.13358 {'hypothetical protein'}
NODE_1684_length_12638_cov_2.13692 {'hypothetical protein'}
NODE_1721_length_12480_cov_1.67433 {'hypothetical protein'}
NODE_1728_length_12452_cov_2.13128 {'hypothetical protein'}
NODE_1811_length_12024_cov_1.84441 {'hypothetical protein'}
NODE_1812_length_12020_cov_1.93652 {'hypothetical protein'}
NODE_1824_length_11968_cov_1.84976 {'hypothetical protein'}
NODE_1827_length_11962_cov_1.99011 {'hypo

In [44]:
list(annotation_gtdb)[100:]

['Putative helicase',
 'Uncharacterized SAM-dependent O-methyltransferase',
 'Probable pyruvate carboxylase',
 'Transcriptional regulator, LysR family',
 'Two-component transcriptional response regulator, LuxR family',
 'Putative two-component sensor',
 'Adenylylsulfate kinase (EC 2.7.1.25)',
 'Putative glycosyl transferase',
 'Export ABC transporter, ATP-binding protein',
 'UDP-glucose 4-epimerase (EC 5.1.3.2)',
 'General secretion pathway protein D',
 'Transcriptional regulator, AraC family',
 'Putative short-chain dehydrogenase',
 'Isocitrate dehydrogenase phosphatase (EC 2.7.11.5)/kinase (EC 3.1.3.-)',
 'Erythronate-4-phosphate dehydrogenase (EC 1.1.1.290)',
 'Transcriptional regulator, HxlR family',
 '3-oxoacyl-[acyl-carrier-protein] synthase, KASII (EC 2.3.1.179)',
 'Phenazine biosynthesis protein PhzF like',
 'Fic domain protein, PA1366 type',
 'Ferrichrome-iron receptor',
 'Iron siderophore receptor protein',
 'Iron siderophore sensor protein',
 'FIG006045: Sigma factor, ECF su

In [ ]:
with open('./data/mmseqs/output/mmseqs_cluster.tsv', 'r') as fh:
    

In [19]:
feature_to_rep_seq_h = {}
ct = 0
with open('./data/mmseqs/output/mmseqs_cluster.tsv', 'r') as fh:
    l = fh.readline()
    a, b = [s.strip() for s in l.split('\t')]
    while l:
        if a:
            feature_to_rep_seq_h[b] = a
        l = fh.readline()
        if l:
            a, b = [s.strip() for s in l.split('\t')]
        ct +=1

In [20]:
len(feature_to_rep_seq_h)

49827277

In [22]:
feature_to_rep_seq_h['96aa2ee242914597e22868592ec54ced543c08570fa136eba8a196d400aab391']

'96aa2ee242914597e22868592ec54ced543c08570fa136eba8a196d400aab391'

In [25]:
feature_to_rep_seq_h['d984f6871448344be069e0ff81ca280bb5fceeb6afafe2af2f1508986cff5bbe']

'bd7c319f4f1b6f455b4e0f39dd1a48b24d4b8a0b5f6e6234246a9fdbc34aeedf'

In [26]:
feature_to_rep_seq_h['bd7c319f4f1b6f455b4e0f39dd1a48b24d4b8a0b5f6e6234246a9fdbc34aeedf']

'bd7c319f4f1b6f455b4e0f39dd1a48b24d4b8a0b5f6e6234246a9fdbc34aeedf'

In [32]:
h = 'bd7c319f4f1b6f455b4e0f39dd1a48b24d4b8a0b5f6e6234246a9fdbc34aeedf'
binary_representation = bin(int(h, 16))[2:].zfill(256)

In [28]:
len(binary_representation)

256

In [41]:
ba = bytes(bytearray.fromhex(h))
ba

b'\xbd|1\x9fO\x1boE[N\x0f9\xdd\x1aH\xb2MK\x8a\x0b_nb4$j\x9f\xdb\xc3J\xee\xdf'

In [44]:
import pyarrow as pa
seq_id = [bytes(bytearray.fromhex(h))]
cluster_id = [bytes(bytearray.fromhex(h))]
table = pa.Table.from_arrays([pa.array(seq_id), pa.array(cluster_id)], names=['cluster_id', 'seq_id'])

In [48]:
df = table.to_pandas()

In [52]:
for row_id, d in df.iterrows():
    print(bytearray(d['cluster_id']).hex())

bd7c319f4f1b6f455b4e0f39dd1a48b24d4b8a0b5f6e6234246a9fdbc34aeedf


In [53]:
from tqdm import tqdm

In [95]:
ws_ids = [155805]
genome_c = set()
for ws_id in ws_ids:
    ws_os = kbase.list_objects(ws_id)
    print(len(ws_os))
    for o in ws_os:
        if o[2].startswith('KBaseGenomes.Genome'):
            genome_c.add(o[1])

679


In [120]:
sfaa2 = SuperFaa()
sfaa2.genomes[genome_id] = genome

In [121]:
sfaa2.aaa()

In [122]:
sfaa2.genome_feature_hash.keys()

dict_keys(['Salt_Pond_MetaG_R2_A_D1_MG_DASTool_bins_concoct_out.84.contigs__.RAST'])

In [123]:
len(sfaa2.genome_feature_hash[genome_id])

3782

In [124]:
len(genome.features)

3787

In [126]:
f.seq

'DTRKMQASYSELCEAASDAVDAVKLEIPEGFRLLKYKLL'

In [169]:
1

1

In [208]:
def _catalog_features(genome_id, genome, c_annotation_database, feature_to_rep_seq_h, score: float, target_genome='self', f_func=None):
    for f in genome.features:
        f_id = f.id
        if f_func:
            f_id = f_func(f)
        if len(f.seq) > 0:
            hseq = HashSeq(f.seq)
            f_hash = hseq.hash_value
            cluster_id = feature_to_rep_seq_h[f_hash]
            if cluster_id not in c_annotation_database:
                c_annotation_database[cluster_id] = {}
            if genome_id not in c_annotation_database[cluster_id]:
                c_annotation_database[cluster_id][genome_id] = {}
            c_annotation_database[cluster_id][genome_id][f'{target_genome}:{f_id}'] = float(score)
def _break_id(f):
    s = f.id.split()
    f_id = s[0]
    return f_id

In [212]:
from tqdm import tqdm

c_annotation_database = {}
from modelseedpy_ext.re.hash_seq import HashSeq
for genome_id in tqdm(genome_c):
    genome = genomes[genome_id]
    _catalog_features(genome_id, genome, c_annotation_database, feature_to_rep_seq_h, 100, 'self')
    for k, v in G[genome_id].items():
        score = v['weight']
        genome_file = h_to_cap.get(k)
        genome = None
        if genome_file:
            if genome_file not in other_genomes:
                other_genomes[genome_file] = MSGenome.from_fasta(base_dir + '/' + genome_file)
            genome = other_genomes[genome_file]
            _catalog_features(genome_id, genome, c_annotation_database, feature_to_rep_seq_h, score, k, _break_id)
            #genome = MSGenome.from_fasta(base_dir + '/' + genome_file)
            #print(genome_id, k, v, h_to_cap[k])
        else:
            genome = genomes.get(k)
            if genome:
                _catalog_features(genome_id, genome, c_annotation_database, feature_to_rep_seq_h, score, k)
            else:
                pass

100%|██████████| 310/310 [57:03<00:00, 11.04s/it]  


In [206]:
len(genome_c), l

(310, '')

In [191]:
len(c_annotation_database)

4467292

In [213]:
with open('./data/mmseqs_ani_prob_v2.json', 'w') as fh:
    fh.write(json.dumps(c_annotation_database))

In [214]:
from modelseedpy.core.msgenome import MSFeature
genome_rep_seqs_features = []
for k in tqdm(c_annotation_database):
    genome_rep_seqs_features.append(MSFeature(k, sfaa.super_faa[k]))
genome_rep_seqs = MSGenome()
genome_rep_seqs.add_features(genome_rep_seqs_features)
genome_rep_seqs.to_fasta('./data/mmseqs_ani_prob_rep_genome_v2.faa')

100%|██████████| 4467292/4467292 [00:17<00:00, 255496.05it/s]


'./data/mmseqs_ani_prob_rep_genome_v2.faa'

In [215]:
c_list = list(c_annotation_database.keys())

In [216]:
c_annotation_database[c_list[0]]

{'Salt_Pond_MetaG_R2_A_D1_MG_DASTool_bins_concoct_out.84.contigs__.RAST': {'self:Salt_Pond_MetaG_R2_A_D1_MG_DASTool_bins_concoct_out.84.contigs__.RAST.CDS.1': 100.0},
 'Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_metabat.15.contigs__.RAST': {'Salt_Pond_MetaG_R2_C_D1_MG_DASTool_bins_metabat.13.contigs.fa_assembly.RAST:Salt_Pond_MetaG_R2_C_D1_MG_DASTool_bins_metabat.13.contigs.fa_assembly.RAST.CDS.4817': 95.7553},
 'Salt_Pond_MetaG_R2_restored_DShore_MG_DASTool_bins_metabat.23.contigs__.RAST': {'Salt_Pond_MetaG_R2_restored_DShore_MG_DASTool_bins_concoct_out.89.contigs__.RAST:Salt_Pond_MetaG_R2_restored_DShore_MG_DASTool_bins_concoct_out.89.contigs__.RAST.CDS.2732': 80.05080000000001},
 'Salt_Pond_MetaG_R1_B_D1_MG_DASTool_bins_metabat.40.contigs__.RAST': {'f2abc695a0ba6d6b155df693471df93f833d74034abcb2a289805d59c36e2f5d:CP081106.1_1075': 77.7569},
 'Salt_Pond_MetaG_R2_A_D1_MG_DASTool_bins_metabat.7.contigs__.RAST': {'Salt_Pond_MetaG_R2_C_D1_MG_DASTool_bins_metabat.13.contigs.fa_assembly.RAST:

In [145]:
base_dir = '/scratch/fliu/data/gtdb/genomes/global/cfs/cdirs/kbase/jungbluth/Projects/Project_Pangenome_GTDB/GTDB_r214_by_spcluster/'
base_cache_dir = '/scratch/fliu/data/kbase/cache/handle/'
for k in everything:
    if k in g_metadata:
        everything[k] = base_cache_dir + '/' +  g_metadata[k][1]
    else:
        p = get_path(k)
        if p:
            everything[k] = base_dir + '/' + p

NameError: name 'everything' is not defined

In [148]:
other_genomes = {}

In [167]:
for genome_id in genome_c:
    for k, v in G[genome_id].items():
        genome_file = h_to_cap.get(k)
        genome = None
        if genome_file:
            if genome_file not in other_genomes:
                other_genomes[genome_file] = MSGenome.from_fasta(base_dir + '/' + genome_file)
            genome = other_genomes[genome_file]
            #genome = MSGenome.from_fasta(base_dir + '/' + genome_file)
            #print(genome_id, k, v, h_to_cap[k])
        else:
            genome = genomes.get(k)
        if genome:
            for f in genome.features:
                
        else:
            pass

...


In [155]:
len(other_genomes)

21173

In [ ]:
for f in genome.features:
    for cluster_id in clusters:
        if cluster_id not in self.annotation_database:
            self.annotation_database[cluster_id] = {}
        if c not in self.annotation_database[cluster_id]:
            self.annotation_database[cluster_id][c] = {}
        self.annotation_database[cluster_id][c][f'self:{f.id}'] = float(100)

In [2]:
import json

In [11]:
with open('/scratch/fliu/data/cliff/mmseqs_ani_prob_v2.json', 'r') as fh:
    data = json.load(fh)

In [13]:
list(data.keys())[:10]

['cd767624267b47c01f08aa9daafdabc32e0d61bffc3236c3f649651db5e534c9',
 'd72e419c02c87d6af954d1fc6148c286683910df15f86b7e0c6ab38bf8ce6ce3',
 '2197c5da174a62b318d27ece2e2781ed23557e1c489e5fb65747a2bb003f64af',
 'c4ad8446e694c50b7afe5ead82855b24b919adcf08e14b22f5bdfe0aa6626ea8',
 '66041d54c267d32643c9720cf9190a405f9a63dff6bbc6dd9f50c111c80ed459',
 'ab45605d36346022bac74fb726c601ff16edee1c72de7639f22e1650cc13a216',
 'b99b4e6b6b6777a8048ebe1a03146b4e5fb968eed06a10351f2ba0cccf4c7d97',
 'd18c864b83fe8bae2cd9f3d7d73f3a20a248c795a96bb281d26e321b7c92c428',
 '85a11820d205a52709d190b038a3b5a8dd5acb3f701c02b7d08f36b8411c4268',
 'b550fd7d11e4532dc51db09cc4c067e31a3ca6d96b37c12fa4495ad318e45624']

In [18]:
protein_clusters_id = 'cd767624267b47c01f08aa9daafdabc32e0d61bffc3236c3f649651db5e534c9'
for k in data[protein_clusters_id]:
    print(k)
    print(data[protein_clusters_id][k])
    print()

Salt_Pond_MetaG_R2_A_D1_MG_DASTool_bins_concoct_out.84.contigs__.RAST
{'self:Salt_Pond_MetaG_R2_A_D1_MG_DASTool_bins_concoct_out.84.contigs__.RAST.CDS.1': 100.0}

Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_metabat.15.contigs__.RAST
{'Salt_Pond_MetaG_R2_C_D1_MG_DASTool_bins_metabat.13.contigs.fa_assembly.RAST:Salt_Pond_MetaG_R2_C_D1_MG_DASTool_bins_metabat.13.contigs.fa_assembly.RAST.CDS.4817': 95.7553}

Salt_Pond_MetaG_R2_restored_DShore_MG_DASTool_bins_metabat.23.contigs__.RAST
{'Salt_Pond_MetaG_R2_restored_DShore_MG_DASTool_bins_concoct_out.89.contigs__.RAST:Salt_Pond_MetaG_R2_restored_DShore_MG_DASTool_bins_concoct_out.89.contigs__.RAST.CDS.2732': 80.05080000000001}

Salt_Pond_MetaG_R1_B_D1_MG_DASTool_bins_metabat.40.contigs__.RAST
{'f2abc695a0ba6d6b155df693471df93f833d74034abcb2a289805d59c36e2f5d:CP081106.1_1075': 77.7569}

Salt_Pond_MetaG_R2_A_D1_MG_DASTool_bins_metabat.7.contigs__.RAST
{'Salt_Pond_MetaG_R2_C_D1_MG_DASTool_bins_metabat.13.contigs.fa_assembly.RAST:Salt_Pond_MetaG_R2_C

In [10]:
data['Salt_Pond_MetaG_R2_C_H2O_MG_DASTool_bins_concoct_out.60.contigs__.RAST']

['Salt_Pond_MetaG_R2_A_H2O_MG_DASTool_bins_concoct_out.51.contigs.fa_assembly.RAST',
 'ede5a1e394f39e509742d01574f8121b1d2c2470ec68f97d86877bb503cc823f',
 'cd3e1d1fcbae456da76bbfc6f3f15e538adcdbdd11c0bcc013bf8667ca9fe042',
 '2a00b09cf8fa0a52c434fbde82e8a9a1d4e967aa340f8d7b93a5f90bc4b5cc81',
 'Salt_Pond_MetaG_R2_B_H2O_MG_DASTool_bins_concoct_out.4.contigs.fa_assembly.RAST',
 'e393c653dde4c5f7feb8d4465dbb2796796fbad404f523a279490d0d8a1e1fc8']

In [8]:
for k in data:
    print(k, len(data[k]))

Salt_Pond_MetaG_R2_C_H2O_MG_DASTool_bins_metabat.29.contigs__.RAST 0
Salt_Pond_MetaG_R2_C_D1_MG_DASTool_bins_concoct_out.7.contigs__.RAST 0
Salt_Pond_MetaG_R2_C_H2O_MG_DASTool_bins_concoct_out.60.contigs__.RAST 6
Salt_Pond_MetaG_R2_restored_DShore_MG_DASTool_bins_metabat.25.contigs__.RAST 1
Salt_Pond_MetaG_R1_B_D1_MG_DASTool_bins_concoct_out.59.contigs__.RAST 1
Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_concoct_out.32.contigs__.RAST 0
Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_metabat.36.contigs__.RAST 1
Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.28.contigs__.RAST 10
Salt_Pond_MetaGSF2_A_H2O_MG_DASTool_bins_concoct_out.33.contigs__.RAST 4
Salt_Pond_MetaG_R1_B_D1_MG_DASTool_bins_metabat.16.contigs__.RAST 0
Salt_Pond_MetaGSF2_A_D2_MG_DASTool_bins_concoct_out.13.contigs__.RAST 10
Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_metabat.48.contigs__.RAST 2
Salt_Pond_MetaG_R1_B_D1_MG_DASTool_bins_metabat.28.contigs__.RAST 0
Salt_Pond_MetaG_R2A_C_D2_MG_DASTool_bins_concoct_out.45.contigs__.RAST 0
Sal